## 0. Task

Exactly 7 prior cases, Micro-F1@7. **Selected:** 8k-word TF-IDF + cite/head fields.


In [ ]:
import os, sys, types, random, json, math, zipfile
from pathlib import Path

_BEFORE = set(globals()) | {"sys", "types", "_BEFORE", "_register"}

def _register(name):
    global _BEFORE
    mod = types.ModuleType(name)
    mod.__name__ = name
    mod.__file__ = name + ".py"
    for key, value in list(globals().items()):
        if key not in _BEFORE:
            setattr(mod, key, value)
    sys.modules[name] = mod
    _BEFORE = set(globals()) | {"sys", "types", "_BEFORE", "_register"}
    print("in this notebook:", name)

print("This notebook is self-contained (starter style). RUN_TRAINING = False for a viva.")


## 1. Lexical first stage + cite/head (selected)


In [ ]:
"""CPU lexical boosters for Phase 7. No GPU, no public checkpoints.

Sweeps TF-IDF / BM25 / head-field / citation overlap on dev, then writes a test zip.
"""
from __future__ import annotations

import json
import os
import re
import zipfile
from pathlib import Path

import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import normalize

ROOT = r"C:\Users\Anshuman Raj\CS781\phase7data\Phase_7_PCR"
OUT = r"C:\Users\Anshuman Raj\CS781\phase7_runs\lexboost"
SUB = r"C:\Users\Anshuman Raj\OneDrive\Desktop\CS781\capstone\phase7\submissions\lexboost"
K = 7
CITE = re.compile(
    r"(?:\b(?:air|scc|scr|supl|suppl)\b\s*\d*|\b(?:section|art(?:icle)?|s)\.?\s*\d+[A-Za-z0-9().-]*"
    r"|\b\d{4}\s+(?:air|scc|scr)\b|\b[A-Za-z][A-Za-z.&' -]{2,60}\bact\b(?:\s*,?\s*\d{4})?)",
    re.I,
)


def join_text(value):
    if isinstance(value, dict):
        value = value.get("text") or ""
    if isinstance(value, list):
        return " ".join(str(part) for part in value)
    return str(value or "")


def clip_words(text, cap, mode="ends"):
    words = text.split()
    if cap <= 0 or len(words) <= cap:
        return text
    if mode == "head":
        return " ".join(words[:cap])
    if mode == "tail":
        return " ".join(words[-cap:])
    head = cap * 2 // 3
    return " ".join(words[:head] + words[-(cap - head):])


def load_split(kind):
    folder = {"train": "Train", "dev": "Dev", "test": "Test"}[kind]
    prefix = kind
    with open(os.path.join(ROOT, folder, f"{prefix}.json"), encoding="utf-8") as handle:
        queries = json.load(handle)
    with open(os.path.join(ROOT, folder, f"{prefix}_candidates.json"), encoding="utf-8") as handle:
        cands = json.load(handle)
    rows = []
    for key, value in queries.items():
        rel = []
        if isinstance(value, dict):
            rel = [str(item) for item in value.get("relevant_candidates") or [] if str(item).strip()]
        rows.append({"id": str(key), "text": join_text(value), "rel": rel})
    pool = {str(key): join_text(value) for key, value in cands.items()}
    return rows, pool


def micro_at_k(queries, ranked, k=K):
    tp = fp = fn = 0
    for row in queries:
        gold, pred = set(row["rel"]), set(ranked[row["id"]][:k])
        tp += len(gold & pred)
        fp += len(pred - gold)
        fn += len(gold - gold & pred) if False else len(gold - pred)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (2 * tp) / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0
    return f1, precision, recall, tp, fp, fn


def rank_from_scores(queries, cand_ids, scores, k=50):
    ranked = {}
    for row, row_scores in zip(queries, scores):
        order = np.argsort(-row_scores)
        out = []
        qid = row["id"]
        for index in order:
            cid = cand_ids[index]
            if cid == qid:
                continue
            out.append(cid)
            if len(out) >= k:
                break
        ranked[qid] = out
    return ranked


def tfidf_scores(qtexts, docs, ngram=2, features=250000, analyzer="word", min_df=2, cap=4000, mode="ends"):
    q_clip = [clip_words(text, cap, mode) for text in qtexts]
    d_clip = [clip_words(text, cap, mode) for text in docs]
    kwargs = dict(
        ngram_range=(1, ngram) if analyzer == "word" else (3, 5),
        max_features=features,
        min_df=min_df,
        max_df=0.95,
        sublinear_tf=True,
        lowercase=True,
        dtype=np.float32,
        token_pattern=r"(?u)\b\w+\b",
        analyzer=analyzer,
    )
    if analyzer != "word":
        kwargs.pop("token_pattern")
        kwargs["ngram_range"] = (3, 5)
        kwargs["analyzer"] = "char_wb"
        kwargs["min_df"] = max(min_df, 3)
    vectorizer = TfidfVectorizer(**kwargs)
    matrix = normalize(vectorizer.fit_transform(d_clip + q_clip))
    return (matrix[len(docs):] @ matrix[:len(docs)].T).toarray()


def bm25_scores(qtexts, docs, k1=1.5, b=0.75, ngram=2, features=250000, cap=4000, mode="ends"):
    q_clip = [clip_words(text, cap, mode) for text in qtexts]
    d_clip = [clip_words(text, cap, mode) for text in docs]
    vectorizer = CountVectorizer(
        ngram_range=(1, ngram),
        max_features=features,
        min_df=2,
        max_df=0.95,
        lowercase=True,
        token_pattern=r"(?u)\b\w+\b",
        dtype=np.float32,
    )
    doc_x = vectorizer.fit_transform(d_clip)
    query_x = vectorizer.transform(q_clip)
    df = np.asarray((doc_x > 0).sum(axis=0)).ravel()
    n_docs = doc_x.shape[0]
    idf = np.log((n_docs - df + 0.5) / (df + 0.5) + 1.0).astype(np.float32)
    dl = np.asarray(doc_x.sum(axis=1)).ravel()
    avgdl = float(dl.mean()) or 1.0
    denom = k1 * (1.0 - b + b * (dl / avgdl))
    doc_x = doc_x.tocsr()
    indptr = doc_x.indptr
    weighted = doc_x.copy()
    for row in range(n_docs):
        start, end = indptr[row], indptr[row + 1]
        tf = doc_x.data[start:end]
        weighted.data[start:end] = idf[doc_x.indices[start:end]] * tf * (k1 + 1.0) / (tf + denom[row])
    return (query_x @ weighted.T).toarray()


def cite_scores(qtexts, docs):
    q_sets = [set(m.lower() for m in CITE.findall(text)) for text in qtexts]
    d_sets = [set(m.lower() for m in CITE.findall(text)) for text in docs]
    out = np.zeros((len(qtexts), len(docs)), dtype=np.float32)
    for i, qset in enumerate(q_sets):
        if not qset:
            continue
        for j, dset in enumerate(d_sets):
            if dset:
                inter = len(qset & dset)
                if inter:
                    out[i, j] = inter / np.sqrt(len(qset) * len(dset))
    return out


def write_zip(path, ranked):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    pred = {qid: ids[:K] for qid, ids in ranked.items()}
    pred_path = os.path.join(os.path.dirname(path), "prediction.json")
    with open(pred_path, "w", encoding="utf-8") as handle:
        json.dump(pred, handle)
    with zipfile.ZipFile(path, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
        bundle.write(pred_path, arcname="prediction.json")
    print("zip", path, "queries", len(pred), flush=True)


def eval_print(name, queries, cand_ids, scores):
    ranked = rank_from_scores(queries, cand_ids, scores)
    f1, precision, recall, tp, fp, fn = micro_at_k(queries, ranked)
    print(f"{name:28s} F1={f1:.4f} P={precision:.4f} R={recall:.4f} tp={tp}", flush=True)
    return f1, ranked, scores


def zscore(mat):
    mean = mat.mean(axis=1, keepdims=True)
    std = mat.std(axis=1, keepdims=True)
    std[std < 1e-6] = 1.0
    return (mat - mean) / std


def main():
    os.makedirs(OUT, exist_ok=True)
    os.makedirs(SUB, exist_ok=True)
    print("loading dev", flush=True)
    dev_q, dev_c = load_split("dev")
    cand_ids = list(dev_c)
    docs = [dev_c[cid] for cid in cand_ids]
    qtexts = [row["text"] for row in dev_q]
    print("dev", len(dev_q), len(docs), flush=True)

    variants = {}
    specs = [
        ("tfidf_w2_3k", dict(ngram=2, cap=3000, mode="ends", features=200000)),
        ("tfidf_w2_5k", dict(ngram=2, cap=5000, mode="ends", features=250000)),
        ("tfidf_w3_5k", dict(ngram=3, cap=5000, mode="ends", features=300000)),
        ("tfidf_head1200", dict(ngram=2, cap=1200, mode="head", features=150000)),
        ("tfidf_head800", dict(ngram=2, cap=800, mode="head", features=120000)),
        ("tfidf_char_head", dict(ngram=2, cap=1500, mode="head", features=180000, analyzer="char_wb")),
        ("bm25_w2_5k", None),
        ("bm25_head1200", None),
        ("cite", None),
    ]
    for name, spec in specs:
        print("scoring", name, flush=True)
        if name.startswith("tfidf"):
            scores = tfidf_scores(qtexts, docs, **spec)
        elif name == "bm25_w2_5k":
            scores = bm25_scores(qtexts, docs, cap=5000, mode="ends")
        elif name == "bm25_head1200":
            scores = bm25_scores(qtexts, docs, cap=1200, mode="head")
        else:
            scores = cite_scores(qtexts, docs)
        f1, ranked, scores = eval_print(name, dev_q, cand_ids, scores)
        variants[name] = {"f1": f1, "scores": scores, "ranked": ranked}

    print("blending", flush=True)
    keys = list(variants)
    zs = {name: zscore(variants[name]["scores"]) for name in keys}
    best = {"f1": -1, "weights": None, "name": None, "scores": None}
    # greedy: start from best single, add others
    order = sorted(keys, key=lambda name: variants[name]["f1"], reverse=True)
    current = zs[order[0]].copy()
    weights = {order[0]: 1.0}
    f1, _, _ = eval_print("start " + order[0], dev_q, cand_ids, current)
    best = {"f1": f1, "weights": dict(weights), "scores": current.copy()}
    for name in order[1:]:
        trial_best = None
        for coef in (0.15, 0.25, 0.4, 0.6, 0.9, 1.2):
            trial = current + coef * zs[name]
            f1, _, _ = eval_print(f" +{coef} {name}", dev_q, cand_ids, trial)
            if f1 > (trial_best[0] if trial_best else -1):
                trial_best = (f1, coef, trial)
        if trial_best and trial_best[0] > best["f1"] + 0.0005:
            current = trial_best[2]
            weights[name] = trial_best[1]
            best = {"f1": trial_best[0], "weights": dict(weights), "scores": current.copy()}
            print("keep", name, trial_best[1], "F1", trial_best[0], flush=True)
        else:
            print("skip", name, flush=True)

    print("BEST DEV", best["f1"], best["weights"], flush=True)
    with open(os.path.join(OUT, "blend.json"), "w", encoding="utf-8") as handle:
        json.dump({"f1": best["f1"], "weights": best["weights"],
                   "singles": {name: variants[name]["f1"] for name in keys}}, handle, indent=2)

    print("loading test", flush=True)
    test_q, test_c = load_split("test")
    t_ids = list(test_c)
    t_docs = [test_c[cid] for cid in t_ids]
    t_q = [row["text"] for row in test_q]
    test_scores = None
    for name, coef in best["weights"].items():
        print("test scoring", name, coef, flush=True)
        if name.startswith("tfidf"):
            spec = dict(specs[[item[0] for item in specs].index(name)][1])
            part = tfidf_scores(t_q, t_docs, **spec)
        elif name == "bm25_w2_5k":
            part = bm25_scores(t_q, t_docs, cap=5000, mode="ends")
        elif name == "bm25_head1200":
            part = bm25_scores(t_q, t_docs, cap=1200, mode="head")
        elif name == "cite":
            part = cite_scores(t_q, t_docs)
        else:
            continue
        part = zscore(part)
        test_scores = part * coef if test_scores is None else test_scores + part * coef
    ranked = rank_from_scores(test_q, t_ids, test_scores)
    write_zip(os.path.join(OUT, "submission.zip"), ranked)
    write_zip(os.path.join(SUB, "submission.zip"), ranked)
    with open(os.path.join(OUT, "prediction_dev.json"), "w", encoding="utf-8") as handle:
        json.dump({qid: ids[:K] for qid, ids in rank_from_scores(dev_q, cand_ids, best["scores"]).items()}, handle)


In [ ]:
_register('phase7_lexboost')


In [ ]:
"""Beat 0.4182 without overfitting the 118-query dev set.

Tune on train (827 queries), confirm on dev, then write a test zip.
No GPU. No public checkpoints.
"""
from __future__ import annotations

import json
import os
import re

import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import normalize

from phase7_lexboost import (
    CITE, K, clip_words, load_split, micro_at_k, rank_from_scores, write_zip, zscore, bm25_scores,
)

OUT = r"C:\Users\Anshuman Raj\CS781\phase7_runs\lexv4"
SUB = r"C:\Users\Anshuman Raj\OneDrive\Desktop\CS781\capstone\phase7\submissions\lexv4"
MASK = re.compile(r"<[^>]+>")
HEAD = re.compile(
    r"(?:CITATION|HEADNOTE|ACT|PETITIONER|RESPONDENT|BENCH)\s*:(.{0,2500})",
    re.I | re.S,
)


def strip_mask(text):
    return MASK.sub(" ", text)


def field_boost(text):
    bits = [m.group(0) for m in HEAD.finditer(text)]
    extra = " ".join(bits) if bits else " ".join(text.split()[:400])
    return clip_words(text, 8000) + " " + extra + " " + extra


def show(name, queries, ids, scores):
    ranked = rank_from_scores(queries, ids, scores)
    f1, precision, recall, tp, fp, fn = micro_at_k(queries, ranked)
    print(f"{name:32s} F1={f1:.4f} P={precision:.4f} R={recall:.4f} tp={tp}", flush=True)
    return f1, ranked, scores


def make_vec(ngram=3, features=400000, min_df=2, max_df=0.95, sublinear=True, ngram_range=None):
    return TfidfVectorizer(
        ngram_range=ngram_range or (1, ngram),
        max_features=features,
        min_df=min_df,
        max_df=max_df,
        sublinear_tf=sublinear,
        lowercase=True,
        dtype=np.float32,
        token_pattern=r"(?u)\b\w+\b",
    )


def cosine(vec, qtexts, docs):
    matrix = normalize(vec.transform(list(docs) + list(qtexts)))
    return (matrix[len(docs):] @ matrix[:len(docs)].T).toarray()


def fit_pooled(texts, **kwargs):
    vec = make_vec(**kwargs)
    vec.fit(texts)
    print(" vocab", len(vec.vocabulary_), "docs", len(texts), flush=True)
    return vec


def cite_tfidf(qtexts, docs):
    q = [" ".join(CITE.findall(text)) for text in qtexts]
    d = [" ".join(CITE.findall(text)) for text in docs]
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True, lowercase=True, dtype=np.float32)
    matrix = normalize(vec.fit_transform(d + q))
    return (matrix[len(docs):] @ matrix[:len(docs)].T).toarray()


def main():
    os.makedirs(OUT, exist_ok=True)
    os.makedirs(SUB, exist_ok=True)
    print("load", flush=True)
    train_q, train_c = load_split("train")
    dev_q, dev_c = load_split("dev")
    test_q, test_c = load_split("test")
    print("sizes", len(train_q), len(train_c), len(dev_q), len(dev_c), len(test_q), len(test_c), flush=True)

    train_ids, dev_ids = list(train_c), list(dev_c)
    train_docs_raw = [train_c[cid] for cid in train_ids]
    dev_docs_raw = [dev_c[cid] for cid in dev_ids]
    train_q_raw = [row["text"] for row in train_q]
    dev_q_raw = [row["text"] for row in dev_q]

    pooled_8k = [clip_words(t, 8000) for t in list(train_c.values()) + list(dev_c.values()) + list(test_c.values())]
    print("fit pooled 8k", len(pooled_8k), flush=True)
    vec8 = fit_pooled(pooled_8k, ngram=3, features=400000)
    vec8b = fit_pooled(pooled_8k, ngram=3, features=600000, max_df=0.9)
    vec23 = fit_pooled(pooled_8k, ngram=3, features=400000, ngram_range=(2, 3))
    vec8strip_src = [clip_words(strip_mask(t), 8000) for t in list(train_c.values()) + list(dev_c.values()) + list(test_c.values())]
    vec8s = fit_pooled(vec8strip_src, ngram=3, features=400000)

    variants = {}

    def add(split, name, scores, queries, ids):
        f1, ranked, scores = show(f"{split}/{name}", queries, ids, scores)
        variants.setdefault(name, {})[split] = {"f1": f1, "scores": scores, "ranked": ranked, "ids": ids}
        return f1

    print("=== train/dev singles ===", flush=True)
    plans = [
        ("base8k_local", None),
        ("pooled8k", vec8),
        ("pooled8k_600k", vec8b),
        ("pooled23", vec23),
        ("pooled_strip", vec8s),
        ("qboost", vec8),
    ]
    # local 8k baseline (fit on split only) — the 0.4182 recipe
    tr_q8 = [clip_words(t, 8000) for t in train_q_raw]
    tr_d8 = [clip_words(t, 8000) for t in train_docs_raw]
    dv_q8 = [clip_words(t, 8000) for t in dev_q_raw]
    dv_d8 = [clip_words(t, 8000) for t in dev_docs_raw]
    local_tr = make_vec(3, 400000)
    local_tr.fit(tr_d8)
    add("train", "base8k_local", cosine(local_tr, tr_q8, tr_d8), train_q, train_ids)
    local_dv = make_vec(3, 400000)
    local_dv.fit(dv_d8)
    add("dev", "base8k_local", cosine(local_dv, dv_q8, dv_d8), dev_q, dev_ids)

    add("train", "pooled8k", cosine(vec8, tr_q8, tr_d8), train_q, train_ids)
    add("dev", "pooled8k", cosine(vec8, dv_q8, dv_d8), dev_q, dev_ids)
    add("train", "pooled8k_600k", cosine(vec8b, tr_q8, tr_d8), train_q, train_ids)
    add("dev", "pooled8k_600k", cosine(vec8b, dv_q8, dv_d8), dev_q, dev_ids)
    add("train", "pooled23", cosine(vec23, tr_q8, tr_d8), train_q, train_ids)
    add("dev", "pooled23", cosine(vec23, dv_q8, dv_d8), dev_q, dev_ids)

    tr_qs = [clip_words(strip_mask(t), 8000) for t in train_q_raw]
    tr_ds = [clip_words(strip_mask(t), 8000) for t in train_docs_raw]
    dv_qs = [clip_words(strip_mask(t), 8000) for t in dev_q_raw]
    dv_ds = [clip_words(strip_mask(t), 8000) for t in dev_docs_raw]
    add("train", "pooled_strip", cosine(vec8s, tr_qs, tr_ds), train_q, train_ids)
    add("dev", "pooled_strip", cosine(vec8s, dv_qs, dv_ds), dev_q, dev_ids)

    tr_qb = [field_boost(t) for t in train_q_raw]
    dv_qb = [field_boost(t) for t in dev_q_raw]
    add("train", "qboost", cosine(vec8, tr_qb, tr_d8), train_q, train_ids)
    add("dev", "qboost", cosine(vec8, dv_qb, dv_d8), dev_q, dev_ids)

    print("bm25", flush=True)
    for k1, b in ((1.2, 0.6), (0.9, 0.4), (1.5, 0.75)):
        name = f"bm25_{k1}_{b}"
        add("train", name, bm25_scores(train_q_raw, train_docs_raw, k1=k1, b=b, ngram=3, features=400000, cap=8000), train_q, train_ids)
        add("dev", name, bm25_scores(dev_q_raw, dev_docs_raw, k1=k1, b=b, ngram=3, features=400000, cap=8000), dev_q, dev_ids)

    print("cite", flush=True)
    add("train", "cite", cite_tfidf(train_q_raw, train_docs_raw), train_q, train_ids)
    add("dev", "cite", cite_tfidf(dev_q_raw, dev_docs_raw), dev_q, dev_ids)

    print("7k/9k pooled", flush=True)
    pooled_7k = [clip_words(t, 7000) for t in list(train_c.values()) + list(dev_c.values()) + list(test_c.values())]
    pooled_9k = [clip_words(t, 9000) for t in list(train_c.values()) + list(dev_c.values()) + list(test_c.values())]
    vec7 = fit_pooled(pooled_7k, ngram=3, features=400000)
    vec9 = fit_pooled(pooled_9k, ngram=3, features=450000)
    add("train", "pooled7k", cosine(vec7, [clip_words(t, 7000) for t in train_q_raw], [clip_words(t, 7000) for t in train_docs_raw]), train_q, train_ids)
    add("dev", "pooled7k", cosine(vec7, [clip_words(t, 7000) for t in dev_q_raw], [clip_words(t, 7000) for t in dev_docs_raw]), dev_q, dev_ids)
    add("train", "pooled9k", cosine(vec9, [clip_words(t, 9000) for t in train_q_raw], [clip_words(t, 9000) for t in train_docs_raw]), train_q, train_ids)
    add("dev", "pooled9k", cosine(vec9, [clip_words(t, 9000) for t in dev_q_raw], [clip_words(t, 9000) for t in dev_docs_raw]), dev_q, dev_ids)

    print("=== summary ===", flush=True)
    names = sorted({name for name in variants}, key=lambda n: variants[n].get("train", {}).get("f1", 0), reverse=True)
    for name in names:
        tr = variants[name].get("train", {}).get("f1")
        dv = variants[name].get("dev", {}).get("f1")
        print(f"  {name:24s} train={tr:.4f}  dev={dv:.4f}" if tr is not None and dv is not None else name, flush=True)

    print("blend on TRAIN", flush=True)
    # start from best train single that also has a score matrix
    start = names[0]
    current_tr = zscore(variants[start]["train"]["scores"])
    current_dv = zscore(variants[start]["dev"]["scores"])
    weights = {start: 1.0}
    best_tr = variants[start]["train"]["f1"]
    best_dv = variants[start]["dev"]["f1"]
    for name in names[1:]:
        if "train" not in variants[name] or "dev" not in variants[name]:
            continue
        trial_best = None
        for coef in (0.1, 0.2, 0.35, 0.5, 0.75):
            trial_tr = current_tr + coef * zscore(variants[name]["train"]["scores"])
            f1_tr = show(f"tr +{coef} {name}", train_q, train_ids, trial_tr)[0]
            if f1_tr > (trial_best[0] if trial_best else -1):
                trial_dv = current_dv + coef * zscore(variants[name]["dev"]["scores"])
                f1_dv = show(f"dv +{coef} {name}", dev_q, dev_ids, trial_dv)[0]
                trial_best = (f1_tr, f1_dv, coef, trial_tr, trial_dv)
        # keep if train improves and dev does not collapse below baseline 0.430 by much
        if trial_best and trial_best[0] > best_tr + 0.0008 and trial_best[1] >= min(best_dv, 0.425) - 0.002:
            current_tr, current_dv = trial_best[3], trial_best[4]
            weights[name] = trial_best[2]
            best_tr, best_dv = trial_best[0], trial_best[1]
            print("KEEP", name, trial_best[2], "train", best_tr, "dev", best_dv, flush=True)
        else:
            print("skip", name, flush=True)

    print("CHOSEN", weights, "train", best_tr, "dev", best_dv, flush=True)
    with open(os.path.join(OUT, "choice.json"), "w") as handle:
        json.dump({"weights": weights, "train": best_tr, "dev": best_dv,
                   "singles": {n: {"train": variants[n]["train"]["f1"], "dev": variants[n]["dev"]["f1"]} for n in names if "dev" in variants[n]}}, handle, indent=2)

    print("test", flush=True)
    test_ids = list(test_c)
    test_docs_raw = [test_c[cid] for cid in test_ids]
    test_q_raw = [row["text"] for row in test_q]
    te_d8 = [clip_words(t, 8000) for t in test_docs_raw]
    te_q8 = [clip_words(t, 8000) for t in test_q_raw]
    test_parts = {}
    makers = {
        "base8k_local": None,
        "pooled8k": lambda: cosine(vec8, te_q8, te_d8),
        "pooled8k_600k": lambda: cosine(vec8b, te_q8, te_d8),
        "pooled23": lambda: cosine(vec23, te_q8, te_d8),
        "pooled_strip": lambda: cosine(vec8s, [clip_words(strip_mask(t), 8000) for t in test_q_raw], [clip_words(strip_mask(t), 8000) for t in test_docs_raw]),
        "qboost": lambda: cosine(vec8, [field_boost(t) for t in test_q_raw], te_d8),
        "pooled7k": lambda: cosine(vec7, [clip_words(t, 7000) for t in test_q_raw], [clip_words(t, 7000) for t in test_docs_raw]),
        "pooled9k": lambda: cosine(vec9, [clip_words(t, 9000) for t in test_q_raw], [clip_words(t, 9000) for t in test_docs_raw]),
        "cite": lambda: cite_tfidf(test_q_raw, test_docs_raw),
    }
    for k1, b in ((1.2, 0.6), (0.9, 0.4), (1.5, 0.75)):
        makers[f"bm25_{k1}_{b}"] = (lambda k1=k1, b=b: bm25_scores(test_q_raw, test_docs_raw, k1=k1, b=b, ngram=3, features=400000, cap=8000))

    # fix base8k_local maker
    def local8k_test():
        vec = make_vec(3, 400000)
        vec.fit(te_d8)
        return cosine(vec, te_q8, te_d8)
    makers["base8k_local"] = local8k_test

    test_scores = None
    for name, coef in weights.items():
        print("test part", name, coef, flush=True)
        part = zscore(makers[name]())
        test_scores = part * coef if test_scores is None else test_scores + part * coef
    ranked = rank_from_scores(test_q, test_ids, test_scores)
    write_zip(os.path.join(OUT, "submission.zip"), ranked)
    write_zip(os.path.join(SUB, "submission.zip"), ranked)
    with open(os.path.join(OUT, "prediction_dev.json"), "w") as handle:
        json.dump({qid: ids[:K] for qid, ids in rank_from_scores(dev_q, dev_ids, current_dv).items()}, handle)
    print("done train", best_tr, "dev", best_dv, flush=True)


In [ ]:
_register('phase7_lexv4')


In [ ]:
"""Rerank the winning 8k TF-IDF list. Fit vectorizer on each split's docs+queries.

CPU only. Writes a test zip if dev F1 beats 0.4302.
"""
from __future__ import annotations

import json
import os

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import normalize

from phase7_lexboost import CITE, K, clip_words, load_split, micro_at_k, rank_from_scores, write_zip, zscore
from phase7_lexv4 import cite_tfidf, field_boost

OUT = r"C:\Users\Anshuman Raj\CS781\phase7_runs\rerank"
SUB = r"C:\Users\Anshuman Raj\OneDrive\Desktop\CS781\capstone\phase7\submissions\rerank"
BASE_DEV = 0.430239


def vec_fit(qtexts, docs, ngram=3, features=400000, cap=8000):
    q_clip = [clip_words(t, cap) for t in qtexts]
    d_clip = [clip_words(t, cap) for t in docs]
    vectorizer = TfidfVectorizer(
        ngram_range=(1, ngram), max_features=features, min_df=2, max_df=0.95,
        sublinear_tf=True, lowercase=True, dtype=np.float32, token_pattern=r"(?u)\b\w+\b",
    )
    matrix = normalize(vectorizer.fit_transform(d_clip + q_clip))
    return (matrix[len(docs):] @ matrix[:len(docs)].T).toarray()


def head_fit(qtexts, docs, cap=1200):
    q_clip = [clip_words(t, cap, "head") for t in qtexts]
    d_clip = [clip_words(t, cap, "head") for t in docs]
    vectorizer = TfidfVectorizer(
        ngram_range=(1, 3), max_features=200000, min_df=2, max_df=0.95,
        sublinear_tf=True, lowercase=True, dtype=np.float32, token_pattern=r"(?u)\b\w+\b",
    )
    matrix = normalize(vectorizer.fit_transform(d_clip + q_clip))
    return (matrix[len(docs):] @ matrix[:len(docs)].T).toarray()


def show(name, queries, ids, scores):
    ranked = rank_from_scores(queries, ids, scores)
    f1, precision, recall, tp, fp, fn = micro_at_k(queries, ranked)
    print(f"{name:36s} F1={f1:.4f} P={precision:.4f} R={recall:.4f} tp={tp}", flush=True)
    return f1, ranked, scores


def blend(a, b, coef):
    return zscore(a) + coef * zscore(b)


def ltr_rerank(queries, ids, base, extra, topn=25, C=1.0):
    """Train logistic on train split extras, apply as additive rerank of topn."""
    # extra: list of score matrices aligned with base
    n_f = 1 + len(extra)
    X, y = [], []
    for i, row in enumerate(queries):
        gold = set(row["rel"])
        if not gold:
            continue
        order = np.argsort(-base[i])
        pos = [j for j in order if ids[j] in gold][:12]
        neg = [j for j in order if ids[j] not in gold and ids[j] != row["id"]][:18]
        for j, label in [(p, 1) for p in pos] + [(n, 0) for n in neg]:
            feats = [base[i, j]] + [mat[i, j] for mat in extra]
            X.append(feats)
            y.append(label)
    X, y = np.asarray(X, dtype=np.float32), np.asarray(y)
    clf = LogisticRegression(C=C, max_iter=200, class_weight="balanced")
    clf.fit(X, y)
    print("ltr coef", clf.coef_, "bias", float(clf.intercept_[0]), "n", len(y), "pos", int(y.sum()), flush=True)
    out = base.copy()
    for i, row in enumerate(queries):
        order = np.argsort(-base[i])[:topn]
        feats = np.stack([base[i, order]] + [mat[i, order] for mat in extra], 1)
        boost = clf.decision_function(feats)
        # keep mass on original ranking, add scaled boost
        new = base[i].copy()
        new[order] = zscore(base[i:i+1, order])[0] + 0.35 * zscore(boost.reshape(1, -1))[0]
        out[i] = new
    return out


def main():
    os.makedirs(OUT, exist_ok=True)
    os.makedirs(SUB, exist_ok=True)
    print("load", flush=True)
    train_q, train_c = load_split("train")
    dev_q, dev_c = load_split("dev")
    test_q, test_c = load_split("test")
    tr_ids, dv_ids, te_ids = list(train_c), list(dev_c), list(test_c)
    tr_docs = [train_c[c] for c in tr_ids]
    dv_docs = [dev_c[c] for c in dv_ids]
    te_docs = [test_c[c] for c in te_ids]
    tr_q = [r["text"] for r in train_q]
    dv_q = [r["text"] for r in dev_q]
    te_q = [r["text"] for r in test_q]

    print("8k base", flush=True)
    tr_base = vec_fit(tr_q, tr_docs)
    dv_base = vec_fit(dv_q, dv_docs)
    te_base = vec_fit(te_q, te_docs)
    show("train/8k", train_q, tr_ids, tr_base)
    f1_8k, _, _ = show("dev/8k", dev_q, dv_ids, dv_base)

    print("aux", flush=True)
    tr_cite, dv_cite, te_cite = cite_tfidf(tr_q, tr_docs), cite_tfidf(dv_q, dv_docs), cite_tfidf(te_q, te_docs)
    tr_head, dv_head, te_head = head_fit(tr_q, tr_docs), head_fit(dv_q, dv_docs), head_fit(te_q, te_docs)
    show("dev/cite", dev_q, dv_ids, dv_cite)
    show("dev/head", dev_q, dv_ids, dv_head)

    print("grid blend on DEV (confirm on train)", flush=True)
    best = {"f1": f1_8k, "name": "8k", "dev": dv_base, "test": te_base}
    for name, tr_m, dv_m, te_m in (("cite", tr_cite, dv_cite, te_cite), ("head", tr_head, dv_head, te_head)):
        for coef in (0.05, 0.08, 0.12, 0.18, 0.25, 0.35):
            dv = blend(dv_base, dv_m, coef)
            f1, _, _ = show(f"dev/8k+{coef}{name}", dev_q, dv_ids, dv)
            if f1 > best["f1"] + 0.0004:
                tr = blend(tr_base, tr_m, coef)
                f1_tr, _, _ = show(f"train/8k+{coef}{name}", train_q, tr_ids, tr)
                best = {"f1": f1, "name": f"8k+{coef}{name}", "dev": dv, "test": blend(te_base, te_m, coef), "train": f1_tr}
                print(" keep", best["name"], "dev", f1, "train", f1_tr, flush=True)

    # both cite and head
    for c_cite in (0.08, 0.12, 0.18):
        for c_head in (0.05, 0.1, 0.18):
            dv = zscore(dv_base) + c_cite * zscore(dv_cite) + c_head * zscore(dv_head)
            f1, _, _ = show(f"dev/8k+{c_cite}c+{c_head}h", dev_q, dv_ids, dv)
            if f1 > best["f1"] + 0.0004:
                tr = zscore(tr_base) + c_cite * zscore(tr_cite) + c_head * zscore(tr_head)
                f1_tr, _, _ = show(f"train/8k+{c_cite}c+{c_head}h", train_q, tr_ids, tr)
                te = zscore(te_base) + c_cite * zscore(te_cite) + c_head * zscore(te_head)
                best = {"f1": f1, "name": f"8k+{c_cite}c+{c_head}h", "dev": dv, "test": te, "train": f1_tr}
                print(" keep", best["name"], "dev", f1, "train", f1_tr, flush=True)

    print("LTR rerank of 8k top-25 using cite+head", flush=True)
    # train LTR on train, apply to all
    def train_ltr(base, cite, head, queries, ids):
        X, y = [], []
        for i, row in enumerate(queries):
            gold = set(row["rel"])
            if not gold:
                continue
            order = np.argsort(-base[i])
            pos = [j for j in order if ids[j] in gold][:12]
            neg = [j for j in order if ids[j] not in gold and ids[j] != row["id"]][:18]
            bz, cz, hz = zscore(base[i:i+1])[0], zscore(cite[i:i+1])[0], zscore(head[i:i+1])[0]
            for j in pos:
                X.append([bz[j], cz[j], hz[j]])
                y.append(1)
            for j in neg:
                X.append([bz[j], cz[j], hz[j]])
                y.append(0)
        clf = LogisticRegression(C=1.0, max_iter=300, class_weight="balanced")
        clf.fit(np.asarray(X, np.float32), np.asarray(y))
        print("ltr w", clf.coef_, flush=True)
        return clf

    clf = train_ltr(tr_base, tr_cite, tr_head, train_q, tr_ids)

    def apply_ltr(base, cite, head, topn=25, mix=0.4):
        out = base.copy()
        for i in range(base.shape[0]):
            order = np.argsort(-base[i])[:topn]
            bz, cz, hz = zscore(base[i:i+1])[0], zscore(cite[i:i+1])[0], zscore(head[i:i+1])[0]
            feats = np.stack([bz[order], cz[order], hz[order]], 1)
            boost = clf.decision_function(feats)
            fused = zscore(base[i:i+1, order])[0] + mix * zscore(boost.reshape(1, -1))[0]
            out[i, order] = fused
        return out

    for mix in (0.2, 0.35, 0.5, 0.7):
        for topn in (15, 25, 40):
            dv = apply_ltr(dv_base, dv_cite, dv_head, topn=topn, mix=mix)
            f1, _, _ = show(f"dev/ltr top{topn} mix{mix}", dev_q, dv_ids, dv)
            if f1 > best["f1"] + 0.0004:
                tr = apply_ltr(tr_base, tr_cite, tr_head, topn=topn, mix=mix)
                f1_tr, _, _ = show(f"train/ltr top{topn} mix{mix}", train_q, tr_ids, tr)
                te = apply_ltr(te_base, te_cite, te_head, topn=topn, mix=mix)
                best = {"f1": f1, "name": f"ltr-{topn}-{mix}", "dev": dv, "test": te, "train": f1_tr}
                print(" keep", best["name"], "dev", f1, "train", f1_tr, flush=True)

    print("BEST", best["name"], "dev", best["f1"], "train", best.get("train"), flush=True)
    with open(os.path.join(OUT, "choice.json"), "w") as handle:
        json.dump({"name": best["name"], "dev": best["f1"], "train": best.get("train"), "base_dev": f1_8k}, handle, indent=2)

    ranked = rank_from_scores(test_q, te_ids, best["test"])
    write_zip(os.path.join(OUT, "submission.zip"), ranked)
    write_zip(os.path.join(SUB, "submission.zip"), ranked)
    with open(os.path.join(OUT, "prediction_dev.json"), "w") as handle:
        json.dump({qid: ids[:K] for qid, ids in rank_from_scores(dev_q, dv_ids, best["dev"]).items()}, handle)
    print("wrote", best["name"], best["f1"], "vs base", f1_8k, flush=True)


In [ ]:
_register('phase7_rerank')


## 2. Serving path


In [ ]:
print("8k TF-IDF word n-grams 1-3, cite/head blend, K=7")
print("prediction.json: {query_id: [id1, ..., id7]}")
